# Clase 220 — Recall@k, Precision@k, MAP@k, NDCG@k, coverage, diversity

Implementación de las 6 métricas + comparativa entre 3 recomendadores sintéticos.

In [ ]:
import numpy as np

def precision_at_k(rel, k):
    """rel: array binario de relevancia en orden de ranking. precision sobre top-k."""
    return rel[:k].sum() / k

def recall_at_k(rel, k, n_relevants):
    return rel[:k].sum() / max(n_relevants, 1)

def ap_at_k(rel, k):
    """Average Precision: precision en posiciones donde hay un relevante, promediada."""
    rel_k = rel[:k]
    if rel_k.sum() == 0: return 0.0
    precisions = [(rel_k[:i+1].sum() / (i+1)) * rel_k[i] for i in range(k)]
    return sum(precisions) / min(k, rel.sum())

def dcg_at_k(rel, k):
    rel_k = rel[:k]
    return float(np.sum(rel_k / np.log2(np.arange(2, k + 2))))

def ndcg_at_k(rel, k):
    ideal = np.sort(rel)[::-1]
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(rel, k) / idcg if idcg > 0 else 0.0

## 1. Ejemplos pedagógicos

In [ ]:
examples = {
    'todos arriba':   np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 0]),
    'todos abajo':    np.array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1]),
    'mezclados':      np.array([1, 0, 1, 0, 1, 0, 0, 0, 0, 0]),
    'ninguno':        np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
}
k = 10
print(f'{"caso":15} {"P@10":>6} {"R@10":>6} {"MAP@10":>8} {"NDCG@10":>8}')
for name, rel in examples.items():
    n_rel = rel.sum()
    print(f'{name:15} {precision_at_k(rel, k):>6.3f} '
          f'{recall_at_k(rel, k, n_rel):>6.3f} '
          f'{ap_at_k(rel, k):>8.3f} '
          f'{ndcg_at_k(rel.astype(float), k):>8.3f}')

print('\n→ "todos arriba" y "todos abajo" tienen igual P@10 y R@10,')
print('  pero distinto MAP y NDCG — porque éstos PENALIZAN posiciones bajas.')

## 2. Comparar 3 recomendadores sintéticos

In [ ]:
rng = np.random.default_rng(42)
n_users, n_items = 200, 500

# Generar test: cada user tiene ~5 items relevantes
R_test = np.zeros((n_users, n_items))
for u in range(n_users):
    relevant = rng.choice(n_items, size=5, replace=False)
    R_test[u, relevant] = 1

# Recomendador 1: random
scores_random = rng.random((n_users, n_items))

# Recomendador 2: popularity (siempre lo mismo)
popularity = rng.random(n_items)
scores_pop = np.tile(popularity, (n_users, 1))

# Recomendador 3: "smart" — un poco mejor que random (perfila correlación con relevancia)
scores_smart = 0.5 * rng.random((n_users, n_items)) + 0.5 * R_test + rng.normal(0, 0.3, R_test.shape)

In [ ]:
def evaluate(scores, R_test, k=10):
    metrics = {'precision': [], 'recall': [], 'map': [], 'ndcg': []}
    for u in range(scores.shape[0]):
        order = np.argsort(-scores[u])[:k]
        rel = R_test[u, order]
        n_rel = R_test[u].sum()
        metrics['precision'].append(precision_at_k(rel, k))
        metrics['recall'].append(recall_at_k(rel, k, n_rel))
        metrics['map'].append(ap_at_k(rel, k))
        metrics['ndcg'].append(ndcg_at_k(rel.astype(float), k))
    return {m: np.mean(v) for m, v in metrics.items()}

import pandas as pd
rows = []
for name, s in [('random', scores_random), ('popularity', scores_pop), ('smart', scores_smart)]:
    m = evaluate(s, R_test, k=10)
    rows.append({'model': name, **m})

results = pd.DataFrame(rows).round(4)
print(results.to_string(index=False))

## 3. Coverage + diversity

In [ ]:
def catalog_coverage(scores, n_items, k=10):
    recommended = set()
    for u in range(scores.shape[0]):
        top = np.argsort(-scores[u])[:k]
        recommended.update(top.tolist())
    return len(recommended) / n_items

# Diversity: 1 - avg pairwise similarity (acá usamos similitud sintética)
from sklearn.metrics.pairwise import cosine_similarity
item_emb = rng.normal(0, 1, (n_items, 16))
item_sim = cosine_similarity(item_emb)

def intra_list_diversity(scores, item_sim, k=10):
    divs = []
    for u in range(scores.shape[0]):
        top = np.argsort(-scores[u])[:k]
        pair_sims = [item_sim[top[i], top[j]] for i in range(k) for j in range(i+1, k)]
        divs.append(1 - np.mean(pair_sims))
    return float(np.mean(divs))

print(f'{"model":12} {"coverage":>10} {"diversity":>11}')
for name, s in [('random', scores_random), ('popularity', scores_pop), ('smart', scores_smart)]:
    c = catalog_coverage(s, n_items, k=10)
    d = intra_list_diversity(s, item_sim, k=10)
    print(f'{name:12} {c:>10.4f} {d:>11.4f}')

print('\n→ Popularity: coverage muy baja (siempre los mismos), diversity también baja.')
print('  Random: coverage perfecta, diversity alta, pero NDCG malísima.')
print('  Smart: balanceado — la idea es maximizar NDCG sin colapsar coverage.')

## 4. Validar contra `recmetrics` si está instalada

In [ ]:
try:
    from sklearn.metrics import ndcg_score
    # Para una user en particular, sklearn devuelve NDCG exacto
    u = 0
    s = scores_smart[u].reshape(1, -1)
    t = R_test[u].reshape(1, -1)
    sk_ndcg = ndcg_score(t, s, k=10)

    # Nuestro NDCG sobre el mismo user
    order = np.argsort(-scores_smart[u])[:10]
    my_ndcg = ndcg_at_k(R_test[u, order].astype(float), 10)
    print(f'sklearn NDCG@10 (user {u}): {sk_ndcg:.4f}')
    print(f'nuestro NDCG@10 (user {u}): {my_ndcg:.4f}')
except ImportError: pass

## Ejercicio guiado

1. Implementá **temporal split**: si tenés timestamps, ordená por fecha; primer 80% train, último 20% test. Compará vs random split — temporal es siempre más pesimista (correcto).
2. Reportá las 6 métricas para los 4 modelos de Clases 216-219 sobre MovieLens 100K. ¿Cuál gana en cada?
3. Novelty: `novelty = mean(-log2(item_popularity))`. Recomendaciones populares: baja novelty.
4. Plot trade-off: NDCG@10 vs coverage para distintos `α` del weighted hybrid (Clase 219).
5. Bonus: si tenés acceso a métricas online (CTR), correlacioná offline NDCG vs CTR. ¿Cuán predictivo es?

## Conclusiones

- Para top-N: NUNCA reportés RMSE/accuracy. Usar recall@k, NDCG@k, MAP@k.
- NDCG es la métrica default — sensible al orden, normalizada [0,1], acepta relevancia graduada.
- Coverage + diversity + novelty son guards: un modelo con NDCG perfecto pero coverage 5% está malo.
- Offline ≈ online pero no idéntico. A/B test (Clase 204) decide en producción.

## ✅ Soluciones de los ejercicios

Implementamos las métricas top-N **desde cero con numpy** y las validamos con casos donde el
valor es evidente. Las métricas de ranking (MAP, NDCG) premian poner lo relevante **arriba**,
no solo incluirlo. Todo ejecutable, sin internet.

### Ejercicio 1 — precision@k y recall@k

Dada la relevancia `[1,0,0,1,0]` y `k=5`: precision@k = relevantes en top-k / k; recall@k =
relevantes en top-k / total relevantes.

In [ ]:
import numpy as np

def precision_at_k(rel, k):
    rel = np.asarray(rel)[:k]
    return rel.sum() / k

def recall_at_k(rel, k, n_relevant):
    rel = np.asarray(rel)[:k]
    return rel.sum() / n_relevant if n_relevant else 0.0

rel = [1, 0, 0, 1, 0]
p5 = precision_at_k(rel, 5)
r5 = recall_at_k(rel, 5, n_relevant=2)
print(f"precision@5 = {p5:.2f} (2 relevantes / 5)")
print(f"recall@5    = {r5:.2f} (2 encontrados / 2 relevantes totales)")

assert abs(p5 - 2/5) < 1e-9 and abs(r5 - 1.0) < 1e-9
assert abs(precision_at_k([1,1,0,0], 2) - 1.0) < 1e-9
print("OK ejercicio 1 — precision@k y recall@k")

### Ejercicio 2 — MAP@k

AP@k promedia la precisión en cada posición donde hay un acierto. Penaliza tener los
relevantes al final: dos listas con la misma precisión@k pueden tener AP distinto.

In [ ]:
def ap_at_k(rel, k):
    rel = np.asarray(rel)[:k]
    if rel.sum() == 0:
        return 0.0
    precisions = [rel[:i+1].sum()/(i+1) for i in range(len(rel)) if rel[i]]
    return float(np.mean(precisions))

def map_at_k(rels, k):
    return float(np.mean([ap_at_k(r, k) for r in rels]))

early = [1, 1, 0, 0, 0]     # relevantes arriba
late  = [0, 0, 0, 1, 1]     # mismos 2 relevantes, pero al final
print(f"precision@5 early={precision_at_k(early,5):.2f}  late={precision_at_k(late,5):.2f} (iguales)")
print(f"AP@5        early={ap_at_k(early,5):.3f}  late={ap_at_k(late,5):.3f} (early gana)")
print(f"MAP@5 de los dos usuarios = {map_at_k([early, late], 5):.3f}")

assert ap_at_k(early, 5) > ap_at_k(late, 5), "AP premia relevantes arriba"
assert precision_at_k(early, 5) == precision_at_k(late, 5), "precision@k no distingue el orden"
print("OK ejercicio 2 — MAP@k penaliza los relevantes tardíos")

### Ejercicio 3 — NDCG@k

DCG descuenta por posición (log2); iDCG es el DCG del orden ideal; NDCG = DCG/iDCG ∈ [0,1].
Mostramos dos listas con **el mismo recall** pero distinto NDCG por el orden.

In [ ]:
def dcg_at_k(rel, k):
    rel = np.asarray(rel, dtype=float)[:k]
    return float((rel / np.log2(np.arange(2, len(rel) + 2))).sum())

def ndcg_at_k(rel, k):
    idcg = dcg_at_k(sorted(rel, reverse=True), k)
    return dcg_at_k(rel, k) / idcg if idcg else 0.0

a = [1, 0, 0, 1]     # relevantes en pos 1 y 4
b = [1, 1, 0, 0]     # relevantes en pos 1 y 2
print(f"recall@4 a={recall_at_k(a,4,2):.2f}  b={recall_at_k(b,4,2):.2f} (iguales)")
print(f"NDCG@4   a={ndcg_at_k(a,4):.3f}  b={ndcg_at_k(b,4):.3f} (b mejor: relevantes más arriba)")

assert abs(recall_at_k(a,4,2) - recall_at_k(b,4,2)) < 1e-9
assert ndcg_at_k(b,4) > ndcg_at_k(a,4)
assert abs(ndcg_at_k([1,1,0,0],4) - 1.0) < 1e-9, "orden ideal -> NDCG=1"
print("OK ejercicio 3 — NDCG distingue el orden aunque el recall sea igual")

### Ejercicio 4 — Leave-one-out por usuario

Para cada usuario con ≥5 ratings dejamos 1 item para test y entrenamos con el resto.
Evaluamos si el modelo lo recupera en el top-k (hit-rate LOO).

In [ ]:
rng = np.random.default_rng(0)
n_users, n_items = 50, 30
R = (rng.random((n_users, n_items)) < 0.2).astype(float)

# un "modelo" de popularidad como ejemplo evaluable
def loo_hit_rate(R, k=10):
    hits, evaluated = 0, 0
    for u in range(n_users):
        pos = np.where(R[u] > 0)[0]
        if len(pos) < 5:
            continue
        held = pos[-1]                        # dejamos el último
        Rtr = R.copy(); Rtr[u, held] = 0
        pop = Rtr.sum(0)                       # popularidad tras quitar el held-out
        pop[Rtr[u] > 0] = -np.inf              # excluir vistos
        topk = np.argsort(pop)[-k:][::-1]
        hits += int(held in topk); evaluated += 1
    return hits / evaluated, evaluated

hr, n_eval = loo_hit_rate(R, k=10)
print(f"usuarios evaluados: {n_eval} | hit-rate@10 LOO = {hr:.3f}")
assert n_eval > 0 and 0.0 <= hr <= 1.0
print("OK ejercicio 4 — protocolo leave-one-out por usuario")

### Ejercicio 5 — Coverage y diversity

**Coverage**: qué % del catálogo aparece en las recomendaciones de todos los usuarios.
**Diversity intra-list**: 1 − similitud media entre los items recomendados a un usuario.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
item_feat = rng.random((n_items, 6))            # features de item para medir diversidad
sim = cosine_similarity(item_feat)

# recomendador de ejemplo: score aleatorio por usuario
scores = rng.random((n_users, n_items))
recs = {u: np.argsort(scores[u])[-10:][::-1] for u in range(n_users)}

recommended = set()
for u in recs: recommended.update(recs[u].tolist())
coverage = len(recommended) / n_items

def intra_list_diversity(items):
    if len(items) < 2: return 0.0
    pairs = [sim[i, j] for a, i in enumerate(items) for j in items[a+1:]]
    return 1 - float(np.mean(pairs))

diversity = float(np.mean([intra_list_diversity(list(recs[u])) for u in recs]))
print(f"coverage del catálogo = {coverage:.1%}")
print(f"diversidad intra-list media = {diversity:.3f}")

assert 0.0 <= coverage <= 1.0
assert 0.0 <= diversity <= 1.0
print("OK ejercicio 5 — coverage y diversity calculadas")